# 03 - Entrenamiento de Faster R-CNN para detección de humo y fuego

Detector de dos etapas (`fasterrcnn_resnet50_fpn_v2` de torchvision) preentrenado
en COCO, con el cabezal ajustado a las clases `smoke` y `fire`.

El código reusable vive en `src/`; este notebook solo arma la configuración,
corre el bucle de épocas y delega el reporte.

## Salidas esperadas

En `reports/results/fasterrcnn_r50fpn/`: `results.csv`, `results.png`,
`confusion_matrix.png`, `confusion_matrix_normalized.png`, `PR_curve.png`,
`F1_curve.png`, `P_curve.png`, `R_curve.png`, `experiment_config_used.yaml`
y `metrics_summary.csv`.

In [ ]:
# ============================================================
# Setup general del entorno
# ============================================================

from pathlib import Path
import os
import sys
import random
import shutil
import time
import yaml

SEED = 42
random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules

print("Ejecutando en Google Colab:", IN_COLAB)
print("Directorio actual:", Path.cwd())

In [ ]:
# ============================================================
# Instalación de dependencias
# ============================================================

REPO_URL = "https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego"
REPO_NAME = "VpC2---Deteccion-de-humo-y-fuego"

if IN_COLAB:
    !pip install -q -r https://raw.githubusercontent.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego/main/requirements.txt

print("Dependencias instaladas.")

In [ ]:
# ============================================================
# Verificación de GPU
# ============================================================

import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DEVICE_NAME = torch.cuda.get_device_name(0)
    print("GPU:", DEVICE_NAME)
else:
    DEVICE = torch.device("cpu")
    DEVICE_NAME = "cpu"
    print("No se detectó GPU. Faster R-CNN en CPU es inviable para 12 épocas.")

print("Device:", DEVICE)

In [ ]:
# ============================================================
# Montar Google Drive
# ============================================================

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/VCII_DFire")
DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / "runs"
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta principal en Drive:", DRIVE_PROJECT_DIR)
print("Carpeta de corridas:", DRIVE_RUNS_DIR)

In [ ]:
# ============================================================
# Clonado o actualización del repositorio
# ============================================================

PROJECT_DIR = Path("/content") / REPO_NAME

if IN_COLAB:
    if PROJECT_DIR.exists():
        print("El repositorio ya existe. Actualizando...")
        %cd {PROJECT_DIR}
        !git pull
    else:
        print("Clonando repositorio...")
        %cd /content
        !git clone {REPO_URL}.git
        %cd {PROJECT_DIR}
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Necesario para que `import src...` funcione.
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR:", PROJECT_DIR)
print("Contenido del proyecto:", os.listdir(PROJECT_DIR))

In [ ]:
# ============================================================
# Carga de configuración del experimento
# ============================================================

EXPERIMENT_CONFIG_PATH = PROJECT_DIR / "configs" / "experiments" / "fasterrcnn_r50fpn.yaml"

if not EXPERIMENT_CONFIG_PATH.exists():
    raise FileNotFoundError(f"No se encontró la configuración: {EXPERIMENT_CONFIG_PATH}")

with open(EXPERIMENT_CONFIG_PATH, "r", encoding="utf-8") as file:
    experiment_config = yaml.safe_load(file)

experiment_name = experiment_config["experiment"]["name"]
training_cfg = experiment_config["training"]

print("Experimento:", experiment_name)
print("Familia:", experiment_config["experiment"]["family"])
print("Modelo:", experiment_config["experiment"]["model"])
print("Épocas:", training_cfg["epochs"], "| batch:", training_cfg["batch"])

In [ ]:
# ============================================================
# Descarga o localización del dataset
# ============================================================

import kagglehub

DATASET_ID = "sayedgamal99/smoke-fire-detection-yolo"
dataset_root = Path(kagglehub.dataset_download(DATASET_ID))


def find_yolo_dataset_dir(root: Path) -> Path:
    """Busca la carpeta con train/images, train/labels, val/images y val/labels."""
    for candidate in [root] + [p for p in root.rglob("*") if p.is_dir()]:
        if all(
            (candidate / split / kind).exists()
            for split in ["train", "val"]
            for kind in ["images", "labels"]
        ):
            return candidate
    raise FileNotFoundError("No se encontró una estructura YOLO válida.")


DATA_DIR = find_yolo_dataset_dir(dataset_root)
print("Carpeta de datos:", DATA_DIR)

In [ ]:
# ============================================================
# Datasets y dataloaders
# ============================================================

from torch.utils.data import DataLoader

from src.data.yolo_dataset import YoloDetectionDataset, collate_fn

train_dataset = YoloDetectionDataset(DATA_DIR / "train", train=True, hflip_prob=0.5, seed=SEED)
val_dataset = YoloDetectionDataset(DATA_DIR / "val", train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=training_cfg["batch"],
    shuffle=True,
    num_workers=training_cfg["workers"],
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=training_cfg["batch"],
    shuffle=False,
    num_workers=training_cfg["workers"],
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
)

print("Imágenes de train:", len(train_dataset))
print("Imágenes de val:  ", len(val_dataset))

# Casi la mitad de las imágenes de train son negativos deliberados.
sin_cajas = sum(1 for i in range(200) if len(train_dataset[i][1]["boxes"]) == 0)
print(f"Negativos en las primeras 200 imágenes: {sin_cajas}")

In [ ]:
# ============================================================
# Modelo, optimizador y scheduler
# ============================================================

from src.engine.trainer import build_optimizer, build_scheduler
from src.modeling.detectors import build_fasterrcnn, count_parameters

torch.manual_seed(SEED)

model = build_fasterrcnn(
    num_classes=3,  # fondo + smoke + fire
    backbone=training_cfg["backbone"],
    trainable_backbone_layers=training_cfg["trainable_backbone_layers"],
    min_size=training_cfg["imgsz"],
    max_size=training_cfg["max_size"],
    pretrained=True,
)
model.to(DEVICE)

optimizer = build_optimizer(model, training_cfg)
scheduler = build_scheduler(optimizer, training_cfg, training_cfg["epochs"])
scaler = torch.amp.GradScaler("cuda") if (training_cfg["amp"] and DEVICE.type == "cuda") else None

print(f"Parámetros entrenables: {count_parameters(model) / 1e6:.2f} M")
print("Optimizador:", type(optimizer).__name__, "| scheduler:", type(scheduler).__name__)
print("AMP:", scaler is not None)

In [ ]:
# ============================================================
# Reanudar desde el último checkpoint si existe
# ============================================================

from src.engine.trainer import load_checkpoint

CHECKPOINT_PATH = DRIVE_RUNS_DIR / experiment_name / "last_checkpoint.pth"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

start_epoch = 0
history = []

if CHECKPOINT_PATH.exists():
    start_epoch, history = load_checkpoint(CHECKPOINT_PATH, model, optimizer, scheduler, DEVICE)
    print(f"Checkpoint encontrado. Reanudando desde la época {start_epoch + 1}.")
else:
    print("No hay checkpoint previo. Entrenamiento desde cero.")

In [ ]:
# ============================================================
# Bucle de entrenamiento
# ============================================================

from src.engine.metrics import collect_predictions, compute_curves, compute_map
from src.engine.trainer import save_checkpoint, train_one_epoch

EPOCHS = training_cfg["epochs"]
elapsed_before = sum(row.get("epoch_time_min", 0.0) for row in history)
start_time = time.time()

for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()
    print(f"\n{'=' * 70}\nÉpoca {epoch + 1}/{EPOCHS}\n{'=' * 70}")

    losses = train_one_epoch(model, optimizer, train_loader, DEVICE, scaler=scaler)

    if scheduler is not None:
        scheduler.step()

    predictions, targets = collect_predictions(model, val_loader, DEVICE)
    map_metrics = compute_map(predictions, targets)
    curves = compute_curves(predictions, targets)

    history.append(
        {
            "epoch": epoch + 1,
            "train/loss_total": losses["loss_total"],
            "train/loss_classifier": losses["loss_classifier"],
            "train/loss_box_reg": losses["loss_box_reg"],
            "train/loss_objectness": losses["loss_objectness"],
            "train/loss_rpn_box_reg": losses["loss_rpn_box_reg"],
            "metrics/mAP50": map_metrics["map50"],
            "metrics/mAP50-95": map_metrics["map50_95"],
            "metrics/precision": curves["best_precision"],
            "metrics/recall": curves["best_recall"],
            "lr": optimizer.param_groups[0]["lr"],
            "epoch_time_min": (time.time() - epoch_start) / 60,
        }
    )

    print(
        f"loss={losses['loss_total']:.4f} | "
        f"mAP50={map_metrics['map50']:.4f} | mAP50-95={map_metrics['map50_95']:.4f}"
    )

    # Se guarda al final de cada época: la sesión de Colab puede cortarse.
    save_checkpoint(CHECKPOINT_PATH, model, optimizer, scheduler, epoch + 1, history)
    print("Checkpoint guardado en:", CHECKPOINT_PATH)

train_time_min = elapsed_before + (time.time() - start_time) / 60
print(f"\nEntrenamiento finalizado. Tiempo total acumulado: {train_time_min:.1f} min")

In [ ]:
# ============================================================
# Guardar los pesos finales en Drive
# ============================================================

if experiment_config["output"]["save_weights"]:
    WEIGHTS_PATH = DRIVE_RUNS_DIR / experiment_name / "best.pth"
    torch.save(model.state_dict(), WEIGHTS_PATH)
    print("Pesos guardados en:", WEIGHTS_PATH)

In [ ]:
# ============================================================
# Reporte completo del experimento
# ============================================================

from src.reporting.experiment_report import generate_experiment_report

REPORTS_RESULTS_DIR = PROJECT_DIR / "reports" / "results" / experiment_name

metrics = generate_experiment_report(
    model=model,
    val_loader=val_loader,
    val_dataset=val_dataset,
    config=experiment_config,
    history=history,
    out_dir=REPORTS_RESULTS_DIR,
    device=DEVICE,
    train_time_min=train_time_min,
    device_name=DEVICE_NAME,
)

import pandas as pd
display(pd.DataFrame([metrics]).T.rename(columns={0: "valor"}))

In [ ]:
# ============================================================
# Visualización de las figuras generadas
# ============================================================

from IPython.display import Image, display

for nombre in [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
]:
    ruta = REPORTS_RESULTS_DIR / nombre
    if ruta.exists():
        print(nombre)
        display(Image(filename=str(ruta)))
    else:
        print("No encontrado:", ruta)

In [ ]:
# ============================================================
# Commit y push de resultados al repositorio
# ============================================================

import subprocess

%cd {PROJECT_DIR}

!git config user.name "Gabriela-Sol"
!git config user.email "solgab.salazar@gmail.com"

pull_result = subprocess.run(
    ["git", "pull", "--rebase", "origin", "main"], text=True, capture_output=True
)
print(pull_result.stdout, pull_result.stderr)

if pull_result.returncode != 0:
    raise RuntimeError("No se pudo completar git pull --rebase. Revisar conflictos.")

for path in [f"reports/results/{experiment_name}/", "configs/experiments/fasterrcnn_r50fpn.yaml"]:
    if Path(path).exists():
        subprocess.run(["git", "add", path], check=True)
        print("Agregado:", path)

status = subprocess.run(["git", "status", "--short"], text=True, capture_output=True)
print(status.stdout)

if not status.stdout.strip():
    print("No hay cambios nuevos para commitear.")
else:
    subprocess.run(
        ["git", "commit", "-m", f"results: update {experiment_name} outputs"], check=True
    )
    print("Commit creado. Para publicarlo: !git push origin main")